# NeuroMera - Audio Emotion Classification (Wav2Vec2)

**How to use:**
1. GPU: `Runtime > Change runtime type > T4 GPU`
2. Update `DATASET_PATH` in Step 1 to your audio folder on Google Drive
3. `Runtime > Run all`

**Audio files must be:** `.wav` format, named like `XXX_XX_EMO_XX.wav` where `EMO` is the emotion code at position 3 (underscore-split).

**Supported emotion codes:** `SAD`, `ANG`, `FEA`, `HAP`, `NEU`, `DIS`

---
## Step 1 - Install & Import

In [ ]:
!pip install -q transformers datasets torchaudio librosa

In [ ]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 - Load Audio Files & Extract Labels

In [ ]:
# ============================================================
#  CHANGE THIS to your audio folder on Google Drive
# ============================================================
DATASET_PATH = "/content/drive/MyDrive/FYPAUDIO/AudioWAV"
MODEL_DIR    = "/content/drive/MyDrive/NeuroMera_Audio_Model"
# ============================================================

# Scan for .wav files
all_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            all_files.append(os.path.join(root, file))

print(f"Total audio files found: {len(all_files)}")

# Extract emotion code from filename (position 3 in underscore-split)
LABEL_MAP = {
    "SAD": "sadness",
    "ANG": "anger",
    "FEA": "fear",
    "HAP": "joy",
    "NEU": "neutral",
    "DIS": "anger",
}

data = []
for path in all_files:
    parts = os.path.basename(path).split("_")
    code = parts[2] if len(parts) > 2 else None
    if code in LABEL_MAP:
        data.append({"path": path, "label": LABEL_MAP[code]})

df = pd.DataFrame(data)
df = df.drop_duplicates()
df = df[df["path"].apply(os.path.exists)]

print(f"Valid labeled files: {len(df)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

---
## Step 3 - Balance & Split

In [ ]:
# Balance: equal samples per emotion
min_count = df["label"].value_counts().min()
df = df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min_count, random_state=42)
).reset_index(drop=True)

print(f"Balanced to {min_count} per label, total: {len(df)}")

# Explicit label mapping
labels_sorted = sorted(df["label"].unique())
LABEL2ID = {label: i for i, label in enumerate(labels_sorted)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
print(f"Label mapping: {LABEL2ID}")

# Split
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

---
## Step 4 - Processor, Dataset & Collator

In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(row["path"], sr=16000)
        return {
            "input_values": audio,
            "labels": LABEL2ID[row["label"]],
        }


@dataclass
class DataCollatorSpeech:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_values = [f["input_values"] for f in features]
        labels = [f["labels"] for f in features]

        batch = self.processor(
            input_values,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


train_dataset = AudioDataset(train_df)
test_dataset  = AudioDataset(test_df)
data_collator = DataCollatorSpeech(processor=processor)

print("Datasets & collator ready.")

---
## Step 5 - Train (Phase 1: frozen backbone)

Only the classification head learns first. Takes ~10 min on T4.

In [ ]:
from transformers import EarlyStoppingCallback

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# Freeze the wav2vec2 backbone — only train the classifier head
for param in model.wav2vec2.parameters():
    param.requires_grad = False

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(pred.label_ids, preds)}

training_args_p1 = TrainingArguments(
    output_dir="./audio_results_p1",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    weight_decay=0.01,                # L2 regularization to prevent memorization
    warmup_ratio=0.1,                 # gradual LR ramp-up for stable training
    label_smoothing_factor=0.1,       # prevents overconfident predictions (key fix)
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # select by lowest loss, not highest accuracy
    greater_is_better=False,           # lower loss = better
    fp16=True,
    fp16_full_eval=False,
    report_to="none",
)

trainer_p1 = Trainer(
    model=model,
    args=training_args_p1,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop when val loss plateaus
)

print("Phase 1: Training classifier head...")
trainer_p1.train()

---
## Step 6 - Train (Phase 2: full fine-tuning)

Unfreeze the backbone and fine-tune everything with a smaller learning rate. Takes ~20 min on T4.

In [ ]:
# Unfreeze all parameters
for param in model.wav2vec2.parameters():
    param.requires_grad = True

# NEW Trainer with NEW args — this is required for the new lr to take effect
training_args_p2 = TrainingArguments(
    output_dir="./audio_results_p2",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,                # L2 regularization to prevent memorization
    warmup_ratio=0.1,                 # gradual LR ramp-up for stable training
    label_smoothing_factor=0.1,       # prevents overconfident predictions (key fix)
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # select by lowest loss, not highest accuracy
    greater_is_better=False,           # lower loss = better
    fp16=True,
    fp16_full_eval=False,
    report_to="none",
)

trainer_p2 = Trainer(
    model=model,
    args=training_args_p2,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop when val loss plateaus
)

print("Phase 2: Fine-tuning full model...")
trainer_p2.train()

---
## Step 7 - Evaluate

In [ ]:
preds_output = trainer_p2.predict(test_dataset)
y_pred = preds_output.predictions.argmax(-1)
y_true = [test_dataset[i]["labels"] for i in range(len(test_dataset))]

print(classification_report(y_true, y_pred, target_names=list(LABEL2ID.keys())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

---
## Step 8 - Save Model to Google Drive

In [ ]:
model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

print(f"Model saved to {MODEL_DIR}")

---
## Step 9 - Load & Predict

Run from here to use an already-trained model without retraining.

In [ ]:
import torch
import librosa
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor

MODEL_DIR = "/content/drive/MyDrive/NeuroMera_Audio_Model"

model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR)
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_emotion(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return model.config.id2label[pred]

print("Model loaded and ready.")

In [ ]:
# Test on a file — change this path to any .wav file
test_file = "/content/drive/MyDrive/FYPAUDIO/AudioWAV/test.wav"  # CHANGE THIS

if os.path.exists(test_file):
    print(f"Predicted Emotion: {predict_emotion(test_file)}")
else:
    print(f"File not found: {test_file}")
    print("Update test_file path to a .wav file on your Drive.")